# Bus Booking Analytics - ETL Pipeline (Tableau Version)

This notebook implements the complete **ETL (Extract, Transform, Load)** pipeline as described in the project documentation for Tableau.
It covers:
1. **Extract**: Loading raw dirty CSV datasets (`bus_booking_raw.csv`, `customers.csv`, `buses.csv`, `routes.csv`) from the local directories.
2. **Transform & Validate**:
   - Flexible Date Parsing using `format='mixed'` to resolve inconsistent date formats.
   - Deduplication (based on `Booking_ID`)
   - Handling Null Values (e.g. dropping bookings with null fares, filling missing customer phone numbers with 'Unknown')
   - Data Standardization (dates formatted to YYYY-MM-DD, proper title casing for names and routes)
   - Data Consistency Checks (Travel Date >= Booking Date, Fare > 0, Seat Number within physical Bus Capacity)
   - Referential Integrity checks between all foreign keys.
3. **Load**: Loading validated data directly into **MySQL** database (`bus_booking_tableau`).

### Step 1: Extract (Load Raw CSVs)

In [1]:
import pandas as pd
import os
import sys

# Import etl_pipeline module from current directory
import etl_pipeline

data_dir = '../data'
bookings, customers, buses, routes = etl_pipeline.load_csv_data(data_dir)
print(f"Raw Bookings count: {len(bookings)}")
print(f"Raw Customers count: {len(customers)}")

### Step 2: Transform, Clean & Validate
We run the validation logic that details formatting repairs, drop counts, and deduplications.

In [2]:
bookings_clean, customers_clean, buses_clean, routes_clean = etl_pipeline.transform_and_validate(
    bookings, customers, buses, routes
)

### Step 3: Load into MySQL Database
Enter your MySQL connection credentials below to load the clean datasets directly into the database.

In [3]:
mysql_config = {
    'host': 'localhost',
    'user': 'root',
    'password': '',  # Update with your MySQL password
    'database': 'bus_booking_tableau'
}

etl_pipeline.load_to_mysql(bookings_clean, customers_clean, buses_clean, routes_clean, mysql_config)

### Step 4: Verification Queries via SQL
Let's run queries on our cleaned dataset in the MySQL database to generate Key Business Metrics.

In [4]:
import warnings
warnings.filterwarnings('ignore', category=UserWarning)

import mysql.connector

conn = mysql.connector.connect(
    host=mysql_config['host'],
    user=mysql_config['user'],
    password=mysql_config['password'],
    database=mysql_config['database']
)

print("--- 1. KEY METRICS SUMMARY ---")
metrics_query = """
SELECT 
    COUNT(Booking_ID) as Total_Bookings,
    ROUND(SUM(Fare_Amount), 2) as Total_Revenue,
    ROUND(AVG(Fare_Amount), 2) as Avg_Fare
FROM Bookings
"""
print(pd.read_sql_query(metrics_query, conn))

print("\n--- 2. TOP 5 ROUTES BY REVENUE ---")
routes_query = """
SELECT 
    r.Source, 
    r.Destination, 
    COUNT(b.Booking_ID) as Booking_Count,
    SUM(b.Fare_Amount) as Route_Revenue
FROM Bookings b
JOIN Routes r ON b.Route_ID = r.Route_ID
GROUP BY r.Route_ID
ORDER BY Route_Revenue DESC
LIMIT 5;
"""
print(pd.read_sql_query(routes_query, conn))

print("\n--- 3. REVENUE BY BUS TYPE ---")
bus_query = """
SELECT 
    bus.Bus_Type, 
    COUNT(b.Booking_ID) as Booking_Count,
    SUM(b.Fare_Amount) as Revenue
FROM Bookings b
JOIN Buses bus ON b.Bus_ID = bus.Bus_ID
GROUP BY bus.Bus_Type
ORDER BY Revenue DESC;
"""
print(pd.read_sql_query(bus_query, conn))

print("\n--- 4. OCCUPANCY RATE ESTIMATE PER BUS ---")
occupancy_query = """
SELECT 
    b.Bus_ID,
    bus.Bus_Number,
    bus.Capacity,
    COUNT(b.Booking_ID) as Bookings_Made,
    ROUND((COUNT(b.Booking_ID) * 100.0) / (bus.Capacity * 20.0), 2) as Estimated_Occupancy_Percentage
FROM Bookings b
JOIN Buses bus ON b.Bus_ID = bus.Bus_ID
GROUP BY b.Bus_ID
ORDER BY Estimated_Occupancy_Percentage DESC;
"""
print(pd.read_sql_query(occupancy_query, conn))

conn.close()